# 4 · Compare everything

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emrtunali/yetkin/blob/main/notebooks/04_compare.ipynb)

Scores every prediction file in Drive on the test set: strict and loose F1, 95 % intervals over postings, a paired
bootstrap against the best system and the spread across seeds. No GPU needed.

In [ ]:
!pip -q install "yetkin @ git+https://github.com/emrtunali/yetkin.git" matplotlib
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/yetkin"

In [ ]:
# the no-learning floor: a dictionary of every span seen in training
from yetkin.baseline import Lexicon
from yetkin.data import load
from yetkin.io import save_predictions
train, test = load("train"), load("test")
lex = Lexicon(train)
save_predictions(f"{OUT}/predictions/lexicon.jsonl", test, [lex.predict(s.tokens) for s in test],
                 meta={"name": "Lexicon (no learning)", "system": "Lexicon (no learning)", "entries": len(lex.entries)})

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display
from yetkin.compare import compare
files = sorted(p for p in Path(f"{OUT}/predictions").glob("*.jsonl") if not p.name.startswith("."))
compare(files, out=Path(f"{OUT}/results"))
display(Markdown(open(f"{OUT}/results/RESULTS.md").read()))
display(Image(f"{OUT}/results/f1.png"))

In [ ]:
# everything in one file to bring back to the repository
!cd "$OUT" && zip -qr /content/yetkin-results.zip predictions results models/*/run.json models/*/train_info.json 2>/dev/null; ls -la /content/yetkin-results.zip
from google.colab import files
files.download("/content/yetkin-results.zip")